# Camera Rotator Angle Check (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-09-02  
**Last Modified:** 2026-09-02  
**Status:** Draft  
**Keywords:** camera rotator, physical_rotator_angle, visitInfo, ConsDB, EFD, MTRotator, sky_rotation  

## Description

Cross-check the **physical camera rotator angle** for LSSTCam visits across the
three independent sources that carry it, and identify visits where the
Consolidated Database (ConsDB) value is missing.

The physical rotator angle (`rotTelPos`) is **not** the sky position angle.  The
raw image `visitInfo` carries two boresight angles, and the physical rotator is
the combination of them:

```
rotator_angle = boresightParAngle - boresightRotAngle - 90 deg      (wrapped to [-180, 180])
                 \_ parallactic _/   \_ sky position angle (ROTPA) _/
```

- `visitInfo.boresightRotAngle` — the **sky** position angle of the camera
  (ROTPA); this is the ConsDB `visit1.sky_rotation` column.
- `visitInfo.boresightParAngle` — the **parallactic** angle of the boresight
  (the angle that is easy to forget).
- ConsDB `visit1_quicklook.physical_rotator_angle` — the canonical mechanical
  rotator angle, which is what this expression reproduces.
- EFD `lsst.sal.MTRotator.rotation.actualPosition` — the rotator encoder
  telemetry, averaged over the exposure.

Key functionality:
1. Pull `physical_rotator_angle` (+ `sky_rotation`, alt/az, program) from ConsDB
   for one or more `day_obs`, and report which visits are **missing** the angle.
2. Read `boresightParAngle` / `boresightRotAngle` from each raw image's
   `visitInfo` via the Butler and rebuild the physical rotator angle.
3. Read `MTRotator.rotation.actualPosition` from the EFD over each exposure.
4. Compare all three sources (offsets, scatter, wrap failures), and check the
   ConsDB `sky_rotation` column against `visitInfo.boresightRotAngle`.
5. Demonstrate the **three-source fallback**: prefer ConsDB, then EFD, then
   Butler `visitInfo`, so a rotator angle is recovered for every visit.

**Output:** Comparison tables printed inline, a summary of ConsDB-missing
visits, four diagnostic plots, and an optional per-visit parquet file.

**Based on:** `ts_intrinsic_wavefront` `intrinsics_lib.get_rotator_data()`
(three-source fallback) and `wcsutils.calc_rotator_from_visitinfo()`.

**Note:** This notebook is deliberately **self-contained** — it imports only
LSST Science Pipelines / `summit_utils` and standard scientific Python, no
personal repository code, so it can be shared directly with Rubin colleagues.


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-09-02 | Aaron Roodman | Initial version |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
   - [4.1 ConsDB visits](#data-consdb)
   - [4.2 Missing ConsDB rotator angles](#data-missing)
   - [4.3 Butler visitInfo angles](#data-visitinfo)
   - [4.4 EFD MTRotator telemetry](#data-efd)
5. [Analysis](#analysis)
   - [5.1 ConsDB vs visitInfo](#analysis-vi)
   - [5.2 Sky angle check](#analysis-sky)
   - [5.3 EFD cross-check](#analysis-efd)
   - [5.4 Three-source fallback](#analysis-fallback)
6. [Results & Plots](#results)


<a id='params'></a>
## Parameters


In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_list = [20260709]           # one or more observation dates (int YYYYMMDD)
instrument = 'lsstcam'              # ConsDB schema: 'lsstcam' | 'lsstcomcam'
butler_instrument = 'LSSTCam'       # Butler instrument name
butler_repo = '/repo/main'          # Butler repository
raw_collection = 'LSSTCam/raw/all'  # collection holding the raw exposures

consdb_url = 'auto'                 # 'auto' picks in-pod vs external endpoint
efd_name = 'usdf_efd'               # EFD instance for makeEfdClient

# Visit selection
seq_num_range = None                # (min, max) inclusive, or None for the whole night
image_types = None                  # e.g. ['science', 'acq'], or None for all
max_visits = 200                    # cap on visits probed (Butler/EFD calls are slow)

# visitInfo access: try these detectors in turn (a raw may be missing one)
visitinfo_detectors = [4, 94, 0]

# Comparison settings
check_all_sources = True            # True: query EFD + visitInfo for ALL visits
                                    # (cross-validation). False: only for visits
                                    # where ConsDB is missing the angle.
tol_deg = 0.5                       # |consdb - other| above this is flagged
rotator_threshold = 90.0            # |rotator| beyond this is physically suspect

output_dir = 'output'               # directory for the optional parquet
output_file = None                  # e.g. 'rotator_check_20260709.parquet'; None = no file


<a id='setup'></a>
## Setup & Imports


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta

# LSST imports (available on the RSP)
from lsst.daf.butler import Butler
from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.efdUtils import makeEfdClient

from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

# Plotting defaults (kept local so this notebook has no repo dependencies)
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 12,
    'legend.frameon': False,
})

print(f'day_obs: {day_obs_list}   instrument: {instrument}')


<a id='functions'></a>
## Helper Functions


In [ ]:
def wrap180(x):
    """Wrap an angle in degrees to [-180, 180].

    Parameters
    ----------
    x : float or array-like
        Angle(s) in degrees.

    Returns
    -------
    float or `numpy.ndarray`
        The wrapped angle(s).
    """
    out = (np.asarray(x, dtype=float) + 180.0) % 360.0 - 180.0
    return float(out) if np.ndim(x) == 0 else out


def rotator_from_visitinfo(par_angle_deg, rotpa_deg):
    """Physical camera rotator angle from the two visitInfo boresight angles.

    The mechanical rotator position (``rotTelPos``, the ConsDB
    ``physical_rotator_angle``) is *not* ``boresightRotAngle`` — that is the sky
    position angle (ROTPA).  The physical rotator follows from combining ROTPA
    with the boresight parallactic angle::

        rotator_angle = par_angle - rotpa - 90 deg

    wrapped to [-180, 180] to absorb values near +/-360.  The physical rotator
    has an allowed travel of roughly [-90, 90] degrees.

    Parameters
    ----------
    par_angle_deg : float or array-like
        Boresight parallactic angle, degrees
        (``visitInfo.getBoresightParAngle()``).
    rotpa_deg : float or array-like
        Boresight rotation (sky position) angle, degrees
        (``visitInfo.getBoresightRotAngle()``).

    Returns
    -------
    float or `numpy.ndarray`
        Physical rotator angle in degrees, wrapped to [-180, 180].
    """
    par = np.asarray(par_angle_deg, dtype=float)
    rotpa = np.asarray(rotpa_deg, dtype=float)
    return wrap180(par - rotpa - 90.0)


In [ ]:
IN_POD_CONSDB_URL = 'http://consdb-pq.consdb:8080/consdb'
EXTERNAL_CONSDB_URL = 'https://usdf-rsp.slac.stanford.edu/consdb'


def in_rsp():
    """True when running inside an RSP (Nublado) JupyterLab pod."""
    return os.path.isdir('/etc/nublado')


def make_consdb_client(url='auto', token_file=None):
    """Return a `lsst.summit.utils.ConsDbClient`.

    ``url='auto'`` selects the in-pod host inside the RSP and the public
    token-injected endpoint elsewhere (e.g. an S3DF login node).

    * **In-pod** — ``consdb-pq.consdb`` only resolves inside the RSP pod and
      must bypass the HTTP proxy (otherwise 502), so ``.consdb`` is appended to
      ``$no_proxy``.  No token needed.
    * **External** — the public endpoint needs an RSP access token, taken from
      ``~/.lsst/consdb_token`` (override with ``token_file``) or else
      ``$ACCESS_TOKEN``, and injected as ``https://user:<token>@host/consdb``.

    Parameters
    ----------
    url : `str`
        ConsDB endpoint, or ``'auto'``.
    token_file : `str`, optional
        Path to a file holding an RSP access token.

    Returns
    -------
    `lsst.summit.utils.ConsDbClient`
    """
    if url == 'auto':
        url = IN_POD_CONSDB_URL if in_rsp() else EXTERNAL_CONSDB_URL
    no_proxy = os.environ.get('no_proxy', '')
    if '.consdb' not in no_proxy:
        os.environ['no_proxy'] = (no_proxy + ',.consdb') if no_proxy else '.consdb'
    if '@' not in url and 'consdb-pq.consdb' not in url:
        tf = Path(token_file) if token_file else Path.home() / '.lsst' / 'consdb_token'
        token = tf.read_text().strip() if tf.exists() else os.environ.get('ACCESS_TOKEN')
        if token:
            url = url.replace('://', f'://user:{token}@', 1)
    return ConsDbClient(url)


def fetch_consdb_visits(cdb, day_obs_list, instrument='lsstcam'):
    """Per-visit rotator / sky angles and pointing from ConsDB.

    ``physical_rotator_angle`` lives in ``visit1_quicklook`` and
    ``sky_rotation`` in ``visit1``; a LEFT JOIN is used so visits with no
    quicklook row still appear (with a NaN rotator angle) — those are exactly
    the visits this notebook is looking for.  Exposure ``obs_start`` /
    ``obs_end`` (TAI) come along for the EFD time windows.

    Parameters
    ----------
    cdb : `lsst.summit.utils.ConsDbClient`
        ConsDB client.
    day_obs_list : `list` [`int`]
        Observation dates as YYYYMMDD.
    instrument : `str`
        ConsDB schema suffix, e.g. ``'lsstcam'``.

    Returns
    -------
    `pandas.DataFrame`
        One row per visit, sorted by ``(day_obs, seq_num)``.
    """
    days = ', '.join(str(int(d)) for d in day_obs_list)
    query = f"""
        SELECT v.visit_id, v.day_obs, v.seq_num, v.band, v.physical_filter,
               v.img_type, v.science_program, v.observation_reason, v.target_name,
               v.altitude, v.azimuth, v.airmass, v.sky_rotation,
               v.s_ra, v.s_dec, v.exp_midpt_mjd,
               e.obs_start, e.obs_end,
               ql.physical_rotator_angle
        FROM cdb_{instrument}.visit1 AS v
        LEFT JOIN cdb_{instrument}.visit1_quicklook AS ql
            ON ql.visit_id = v.visit_id
        LEFT JOIN cdb_{instrument}.exposure AS e
            ON e.day_obs = v.day_obs AND e.seq_num = v.seq_num
        WHERE v.day_obs IN ({days})
        ORDER BY v.day_obs, v.seq_num
    """
    df = cdb.query(query).to_pandas()
    for col in ('physical_rotator_angle', 'sky_rotation', 'altitude',
                'azimuth', 'airmass', 's_ra', 's_dec', 'exp_midpt_mjd'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def select_visits(df, seq_num_range=None, image_types=None, max_visits=None):
    """Apply the seq_num / img_type / count cuts from the Parameters cell."""
    sel = df.copy()
    if seq_num_range is not None:
        lo, hi = seq_num_range
        sel = sel[(sel.seq_num >= lo) & (sel.seq_num <= hi)]
    if image_types is not None:
        wanted = {t.lower() for t in image_types}
        sel = sel[sel.img_type.fillna('').str.lower().isin(wanted)]
    sel = sel.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)
    if max_visits is not None and len(sel) > max_visits:
        print(f'  limiting {len(sel)} -> {max_visits} visits (max_visits)')
        sel = sel.iloc[:max_visits].reset_index(drop=True)
    return sel


In [ ]:
def fetch_visitinfo_angles(butler, pairs, instrument='LSSTCam', detectors=(4, 94, 0)):
    """Boresight angles from each raw exposure's ``visitInfo``.

    Uses the ``raw.visitInfo`` *component* get, so no pixels are read.  Each
    detector in ``detectors`` is tried in turn (a given raw may be missing one),
    then a full ``raw`` get as a last resort.

    Parameters
    ----------
    butler : `lsst.daf.butler.Butler`
        Butler with the raw collection.
    pairs : `list` [(`int`, `int`)]
        ``(day_obs, seq_num)`` pairs.
    instrument : `str`
        Butler instrument name.
    detectors : `tuple` [`int`]
        Candidate detector ids.

    Returns
    -------
    `pandas.DataFrame`
        Columns ``day_obs``, ``seq_num``, ``vi_par_angle``, ``vi_rotpa``,
        ``vi_rotator_angle``, ``vi_detector``.  NaN where nothing resolved.
    """
    records = []
    for day_obs_val, seq_num in tqdm(pairs, desc='visitInfo'):
        rec = {'day_obs': day_obs_val, 'seq_num': seq_num,
               'vi_par_angle': np.nan, 'vi_rotpa': np.nan,
               'vi_rotator_angle': np.nan, 'vi_detector': -1}
        for det in detectors:
            kw = dict(instrument=instrument, detector=det,
                      day_obs=day_obs_val, seq_num=seq_num)
            vi = None
            try:
                vi = butler.get('raw.visitInfo', **kw)
            except Exception:
                try:
                    vi = butler.get('raw', **kw).visitInfo
                except Exception:
                    continue
            par = vi.getBoresightParAngle().asDegrees()
            rotpa = vi.getBoresightRotAngle().asDegrees()
            rec.update(vi_par_angle=par, vi_rotpa=rotpa,
                       vi_rotator_angle=rotator_from_visitinfo(par, rotpa),
                       vi_detector=det)
            break
        else:
            print(f'  visitInfo unavailable for day_obs={day_obs_val} '
                  f'seq_num={seq_num} (left NaN)')
        records.append(rec)

    df = pd.DataFrame(records)
    n_ok = int(df.vi_rotator_angle.notna().sum())
    print(f'  visitInfo angles for {n_ok}/{len(df)} visits')
    return df


In [ ]:
EFD_ROTATOR_TOPIC = 'lsst.sal.MTRotator.rotation'


async def fetch_efd_rotator(efd_client, visits, pre_padding=0.0, post_padding=0.0):
    """Mean ``MTRotator.rotation.actualPosition`` over each exposure.

    The exposure window comes from the ConsDB ``obs_start`` / ``obs_end``
    columns (TAI), converted to UTC for the EFD query — so this source needs no
    Butler.  Visits with no ``obs_start`` or no telemetry in the window stay NaN.

    Parameters
    ----------
    efd_client : `lsst_efd_client.EfdClient`
        EFD client from `makeEfdClient`.
    visits : `pandas.DataFrame`
        Must have ``day_obs``, ``seq_num``, ``obs_start``, ``obs_end``.
    pre_padding, post_padding : `float`
        Seconds to widen the window at each end.

    Returns
    -------
    `pandas.DataFrame`
        Columns ``day_obs``, ``seq_num``, ``efd_rotator_angle``,
        ``efd_rotator_std``, ``efd_n_samples``.
    """
    records = []
    for row in tqdm(list(visits.itertuples(index=False)), desc='EFD MTRotator'):
        rec = {'day_obs': int(row.day_obs), 'seq_num': int(row.seq_num),
               'efd_rotator_angle': np.nan, 'efd_rotator_std': np.nan,
               'efd_n_samples': 0}
        try:
            if pd.isna(row.obs_start):
                records.append(rec)
                continue
            begin = Time(str(row.obs_start), scale='tai')
            end = (Time(str(row.obs_end), scale='tai')
                   if not pd.isna(row.obs_end) else begin + TimeDelta(30.0, format='sec'))
            if pre_padding:
                begin = begin - TimeDelta(pre_padding, format='sec')
            if post_padding:
                end = end + TimeDelta(post_padding, format='sec')

            data = await efd_client.select_time_series(
                EFD_ROTATOR_TOPIC, ['actualPosition'], begin.utc, end.utc)
            if data is not None and len(data) and 'actualPosition' in data.columns:
                vals = np.asarray(data['actualPosition'].values, dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size:
                    rec.update(efd_rotator_angle=float(vals.mean()),
                               efd_rotator_std=float(vals.std()),
                               efd_n_samples=int(vals.size))
        except Exception as exc:
            print(f'  EFD failed for day_obs={rec["day_obs"]} '
                  f'seq_num={rec["seq_num"]}: {type(exc).__name__}: {exc}')
        records.append(rec)

    df = pd.DataFrame(records, columns=['day_obs', 'seq_num', 'efd_rotator_angle',
                                        'efd_rotator_std', 'efd_n_samples'])
    n_ok = int(df.efd_rotator_angle.notna().sum())
    print(f'  EFD rotator for {n_ok}/{len(df)} visits')
    return df


async def close_efd_client(efd_client):
    """Close the aiohttp session behind an EFD client, if it exposes one."""
    for attr in ('influx_client', 'client'):
        obj = getattr(efd_client, attr, None)
        close = getattr(obj, 'close', None)
        if close is not None:
            try:
                res = close()
                if hasattr(res, '__await__'):
                    await res
            except Exception:
                pass


In [ ]:
def compare_columns(df, ref_col, other_col, label, tol_deg=0.5):
    """Print offset / scatter statistics for ``other_col`` minus ``ref_col``.

    The difference is wrapped to [-180, 180] before the statistics, so a
    360-degree wrap does not masquerade as a huge disagreement.  A near-zero
    standard deviation with a non-zero mean means the two sources differ by a
    pure constant offset (a convention difference, not noise).

    Parameters
    ----------
    df : `pandas.DataFrame`
        Table holding both columns.
    ref_col, other_col : `str`
        Column names; the difference formed is ``other - ref``.
    label : `str`
        Text used in the printed heading.
    tol_deg : `float`
        Differences above this magnitude are counted as outliers.

    Returns
    -------
    `pandas.Series`
        The wrapped differences (NaN where either input is NaN).
    """
    diff = pd.Series(np.full(len(df), np.nan), index=df.index)
    good = df[ref_col].notna() & df[other_col].notna()
    diff[good] = wrap180(df.loc[good, other_col] - df.loc[good, ref_col])

    n = int(good.sum())
    print(f'\n{label}   ({other_col} - {ref_col})')
    print('-' * 72)
    if n == 0:
        print('  no visits have both values')
        return diff
    d = diff[good]
    n_out = int((d.abs() > tol_deg).sum())
    print(f'  n = {n}    mean = {d.mean():+8.4f} deg    median = {d.median():+8.4f} deg')
    print(f'  std = {d.std():8.4f} deg    min = {d.min():+8.4f}    max = {d.max():+8.4f}')
    print(f'  |diff| > {tol_deg} deg: {n_out}/{n} visits')
    if n_out:
        cols = [c for c in ('day_obs', 'seq_num', 'img_type', 'science_program',
                            ref_col, other_col) if c in df.columns]
        worst = df.loc[d.abs().sort_values(ascending=False).index[:10], cols].copy()
        worst['diff'] = d.loc[worst.index]
        print('  worst offenders:')
        print(worst.to_string(index=False))
    return diff


<a id='data'></a>
## Data Access


<a id='data-consdb'></a>
### 4.1 ConsDB visits

`physical_rotator_angle` from `visit1_quicklook`, `sky_rotation` from `visit1`,
joined LEFT so visits with no quicklook row still appear.


In [ ]:
cdb = make_consdb_client(consdb_url)
visits_all = fetch_consdb_visits(cdb, day_obs_list, instrument=instrument)
print(f'ConsDB: {len(visits_all)} visits for day_obs={day_obs_list}')

visits = select_visits(visits_all, seq_num_range=seq_num_range,
                       image_types=image_types, max_visits=max_visits)
print(f'Selected: {len(visits)} visits')

print('\nimg_type counts:')
print(visits.img_type.fillna('(none)').value_counts().to_string())
visits.head()


<a id='data-missing'></a>
### 4.2 Missing ConsDB rotator angles

Which visits have no `physical_rotator_angle`, broken down by night and by
`img_type` / `science_program` — a missing angle usually tracks a visit with no
`visit1_quicklook` row at all (calibrations, aborted or unprocessed exposures).


In [ ]:
missing_mask = visits.physical_rotator_angle.isna()
n_missing = int(missing_mask.sum())
print(f'ConsDB physical_rotator_angle: {len(visits) - n_missing} present, '
      f'{n_missing} missing  ({100.0 * n_missing / max(len(visits), 1):.1f}%)')

if n_missing:
    print('\nMissing by day_obs:')
    for day in sorted(visits.day_obs.unique()):
        tot = int((visits.day_obs == day).sum())
        mis = int((missing_mask & (visits.day_obs == day)).sum())
        print(f'  day_obs={day}: {mis}/{tot} missing')

    print('\nMissing by img_type:')
    print(visits.loc[missing_mask, 'img_type'].fillna('(none)')
          .value_counts().to_string())

    print('\nMissing by science_program:')
    print(visits.loc[missing_mask, 'science_program'].fillna('(none)')
          .value_counts().to_string())

    cols = ['day_obs', 'seq_num', 'img_type', 'science_program', 'band',
            'sky_rotation', 'altitude']
    show = visits.loc[missing_mask, [c for c in cols if c in visits.columns]]
    print(f'\nFirst {min(20, n_missing)} visits missing the ConsDB rotator angle:')
    print(show.head(20).to_string(index=False))
    if n_missing > 20:
        print(f'  ... and {n_missing - 20} more')
else:
    print('\nEvery selected visit has a ConsDB physical_rotator_angle.')


<a id='data-visitinfo'></a>
### 4.3 Butler visitInfo angles

Read `boresightParAngle` and `boresightRotAngle` from the raw `visitInfo` and
rebuild the physical rotator angle.  With `check_all_sources = True` this runs
over every selected visit (for the cross-check); otherwise only over the visits
ConsDB is missing.


In [ ]:
butler = Butler(butler_repo, instrument=butler_instrument,
                collections=[raw_collection])

probe = visits if check_all_sources else visits[missing_mask]
pairs = [(int(d), int(s)) for d, s in zip(probe.day_obs, probe.seq_num)]
print(f'Querying Butler visitInfo for {len(pairs)} visits '
      f'({"all selected" if check_all_sources else "ConsDB-missing only"})...')

vi_df = (fetch_visitinfo_angles(butler, pairs, instrument=butler_instrument,
                                detectors=visitinfo_detectors)
         if pairs else
         pd.DataFrame(columns=['day_obs', 'seq_num', 'vi_par_angle', 'vi_rotpa',
                               'vi_rotator_angle', 'vi_detector']))
vi_df.head()


<a id='data-efd'></a>
### 4.4 EFD MTRotator telemetry

Mean `actualPosition` over each exposure window (`obs_start` → `obs_end` from
ConsDB, TAI → UTC).  `efd_rotator_std` shows how much the rotator moved during
the exposure — it should be small for a tracking exposure.


In [ ]:
efd_client = makeEfdClient(efd_name) if efd_name else makeEfdClient()

try:
    efd_df = await fetch_efd_rotator(efd_client, probe)
finally:
    await close_efd_client(efd_client)

if len(efd_df) and efd_df.efd_rotator_angle.notna().any():
    ok = efd_df.efd_rotator_std.notna()
    print(f'\nIn-exposure rotator motion (std of actualPosition): '
          f'median {efd_df.loc[ok, "efd_rotator_std"].median():.4f} deg, '
          f'max {efd_df.loc[ok, "efd_rotator_std"].max():.4f} deg')
efd_df.head()


In [ ]:
# One table, all sources
rot = visits.merge(vi_df, on=['day_obs', 'seq_num'], how='left') \
            .merge(efd_df, on=['day_obs', 'seq_num'], how='left')
assert len(rot) == len(visits), 'merge changed the row count'

print(f'{len(rot)} visits with:')
for col, name in [('physical_rotator_angle', 'ConsDB physical_rotator_angle'),
                  ('efd_rotator_angle', 'EFD MTRotator.actualPosition'),
                  ('vi_rotator_angle', 'visitInfo-derived rotator'),
                  ('sky_rotation', 'ConsDB sky_rotation'),
                  ('vi_rotpa', 'visitInfo boresightRotAngle'),
                  ('vi_par_angle', 'visitInfo boresightParAngle')]:
    n = int(rot[col].notna().sum()) if col in rot.columns else 0
    print(f'  {name:<36} {n:4d}')


<a id='analysis'></a>
## Analysis


<a id='analysis-vi'></a>
### 5.1 ConsDB vs visitInfo-derived rotator angle

Is `parAngle - ROTPA - 90` the ConsDB `physical_rotator_angle`?  A small mean
with small scatter confirms the formula; a non-zero mean with ~zero scatter
would mean a constant convention offset.


In [ ]:
rot['diff_vi'] = compare_columns(rot, 'physical_rotator_angle', 'vi_rotator_angle',
                                 'ConsDB vs visitInfo-derived rotator angle',
                                 tol_deg=tol_deg)

# Per-visit table
cols = [c for c in ('day_obs', 'seq_num', 'img_type', 'vi_par_angle', 'vi_rotpa',
                    'sky_rotation', 'vi_rotator_angle', 'physical_rotator_angle',
                    'efd_rotator_angle', 'diff_vi') if c in rot.columns]
print('\nPer-visit angles (first 30):')
print(rot[cols].head(30).to_string(index=False,
      float_format=lambda v: f'{v:9.4f}'))


<a id='analysis-sky'></a>
### 5.2 Sky angle check: ConsDB `sky_rotation` vs `visitInfo.boresightRotAngle`

These should be the *same* quantity (the sky position angle, ROTPA) — a useful
control on the comparison above, and a reminder that neither is the physical
rotator.


In [ ]:
rot['diff_sky'] = compare_columns(rot, 'sky_rotation', 'vi_rotpa',
                                 'ConsDB sky_rotation vs visitInfo boresightRotAngle',
                                 tol_deg=tol_deg)

# Sanity: the sky angle is NOT the physical rotator. Show how different they are.
both = rot.sky_rotation.notna() & rot.physical_rotator_angle.notna()
if both.any():
    d = wrap180(rot.loc[both, 'sky_rotation'] - rot.loc[both, 'physical_rotator_angle'])
    print(f'\nFor reference, sky_rotation - physical_rotator_angle over {int(both.sum())} '
          f'visits:\n  mean {np.mean(d):+.3f} deg, std {np.std(d):.3f} deg, '
          f'range [{np.min(d):+.2f}, {np.max(d):+.2f}] '
          f'-- large and variable, i.e. two different angles.')


<a id='analysis-efd'></a>
### 5.3 EFD cross-check

The rotator encoder is the closest thing to ground truth, so ConsDB should agree
with it very tightly.


In [ ]:
rot['diff_efd'] = compare_columns(rot, 'physical_rotator_angle', 'efd_rotator_angle',
                                  'ConsDB vs EFD MTRotator.actualPosition',
                                  tol_deg=tol_deg)

rot['diff_efd_vi'] = compare_columns(rot, 'efd_rotator_angle', 'vi_rotator_angle',
                                     'EFD vs visitInfo-derived rotator angle',
                                     tol_deg=tol_deg)


<a id='analysis-fallback'></a>
### 5.4 Three-source fallback

Build the best available rotator angle per visit, preferring **ConsDB**, then
**EFD** (fast, and the mechanical truth), then **Butler visitInfo** (slowest,
and a derived quantity).  `rotator_source` records which one was used.


In [ ]:
rot['rotator_angle'] = (rot.physical_rotator_angle
                        .fillna(rot.efd_rotator_angle)
                        .fillna(rot.vi_rotator_angle))

rot['rotator_source'] = np.select(
    [rot.physical_rotator_angle.notna(),
     rot.efd_rotator_angle.notna(),
     rot.vi_rotator_angle.notna()],
    ['consdb', 'efd', 'visitinfo'],
    default='none')

rot['rotator_flagged'] = rot.rotator_angle.abs() > rotator_threshold

print('Rotator angle source used:')
print(rot.rotator_source.value_counts().to_string())

n_none = int((rot.rotator_source == 'none').sum())
n_rec = int(rot.rotator_source.isin(['efd', 'visitinfo']).sum())
print(f'\nRecovered by fallback (ConsDB was missing): {n_rec}')
print(f'Still with no rotator angle from any source: {n_none}')
if n_none:
    cols = ['day_obs', 'seq_num', 'img_type', 'science_program']
    print(rot.loc[rot.rotator_source == 'none',
                  [c for c in cols if c in rot.columns]].to_string(index=False))

n_flag = int(rot.rotator_flagged.sum())
print(f'\n|rotator_angle| > {rotator_threshold} deg (outside physical travel): {n_flag}')
if n_flag:
    print(rot.loc[rot.rotator_flagged,
                  ['day_obs', 'seq_num', 'rotator_angle', 'rotator_source']]
          .to_string(index=False))


<a id='results'></a>
## Results & Plots


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) ConsDB vs visitInfo-derived, with y = x
ax = axes[0, 0]
m = rot.physical_rotator_angle.notna() & rot.vi_rotator_angle.notna()
if m.any():
    ax.scatter(rot.loc[m, 'physical_rotator_angle'], rot.loc[m, 'vi_rotator_angle'],
               s=18, alpha=0.7, label='visitInfo')
    m2 = rot.physical_rotator_angle.notna() & rot.efd_rotator_angle.notna()
    if m2.any():
        ax.scatter(rot.loc[m2, 'physical_rotator_angle'], rot.loc[m2, 'efd_rotator_angle'],
                   s=18, alpha=0.7, marker='x', label='EFD')
    lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lim, lim, 'k--', lw=1, label='y = x')
    ax.legend(loc='best', fontsize=9)
ax.set_xlabel('ConsDB physical_rotator_angle [deg]')
ax.set_ylabel('other source [deg]')
ax.set_title('(a) ConsDB vs the other two sources')

# (b) differences vs seq_num
ax = axes[0, 1]
n_plotted = 0
for col, lbl, mk in [('diff_vi', 'visitInfo - ConsDB', 'o'),
                     ('diff_efd', 'EFD - ConsDB', 'x')]:
    m = rot[col].notna()
    if m.any():
        ax.scatter(rot.loc[m, 'seq_num'], rot.loc[m, col], s=18, alpha=0.7,
                   marker=mk, label=lbl)
        n_plotted += 1
ax.axhline(0, color='k', lw=1)
for lvl in (-tol_deg, tol_deg):
    ax.axhline(lvl, color='r', ls=':', lw=1)
ax.set_xlabel('seq_num')
ax.set_ylabel('difference [deg]')
ax.set_title(f'(b) Difference vs seq_num (dotted = +/-{tol_deg} deg)')
if n_plotted:
    ax.legend(loc='best', fontsize=9)

# (c) histogram of the visitInfo - ConsDB difference
ax = axes[1, 0]
d = rot.diff_vi.dropna()
if len(d):
    ax.hist(d, bins=min(40, max(8, len(d) // 3)), alpha=0.8)
    ax.axvline(d.mean(), color='r', ls='--', lw=1.2,
               label=f'mean {d.mean():+.3f} deg')
    ax.axvline(0, color='k', lw=1)
    ax.legend(loc='best', fontsize=9)
    ax.set_title(f'(c) visitInfo - ConsDB   (std {d.std():.3f} deg, n={len(d)})')
else:
    ax.set_title('(c) visitInfo - ConsDB   (no overlap)')
ax.set_xlabel('difference [deg]')
ax.set_ylabel('visits')

# (d) the three angles vs seq_num, plus the sky angle for contrast
ax = axes[1, 1]
for col, lbl, sty in [('physical_rotator_angle', 'ConsDB physical rotator', '-o'),
                      ('efd_rotator_angle', 'EFD actualPosition', '--x'),
                      ('vi_rotator_angle', 'visitInfo parAng-ROTPA-90', ':s'),
                      ('sky_rotation', 'ConsDB sky_rotation (sky PA)', '-.')]:
    if col in rot.columns and rot[col].notna().any():
        sub = rot[rot[col].notna()].sort_values('seq_num')
        ax.plot(sub.seq_num, sub[col], sty, ms=4, lw=1, alpha=0.85, label=lbl)
missed = rot[rot.physical_rotator_angle.isna()]
if len(missed):
    ax.scatter(missed.seq_num, np.zeros(len(missed)), marker='|', s=200,
               color='red', label=f'ConsDB missing ({len(missed)})')
ax.set_xlabel('seq_num')
ax.set_ylabel('angle [deg]')
ax.set_title('(d) Angles vs seq_num')
ax.legend(loc='best', fontsize=8)

fig.suptitle(f'Camera rotator angle check — day_obs {day_obs_list}, '
             f'{len(rot)} visits', fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Summary
# ============================================================
print('=' * 72)
print(f'Camera rotator angle check — day_obs {day_obs_list}, {len(rot)} visits')
print('=' * 72)

n_mis = int(rot.physical_rotator_angle.isna().sum())
print(f'\nConsDB physical_rotator_angle missing : {n_mis}/{len(rot)}')
print(f'Recovered from EFD                    : '
      f'{int((rot.rotator_source == "efd").sum())}')
print(f'Recovered from Butler visitInfo       : '
      f'{int((rot.rotator_source == "visitinfo").sum())}')
print(f'No angle from any source              : '
      f'{int((rot.rotator_source == "none").sum())}')

for col, lbl in [('diff_vi', 'visitInfo - ConsDB'),
                 ('diff_efd', 'EFD       - ConsDB'),
                 ('diff_efd_vi', 'visitInfo - EFD   '),
                 ('diff_sky', 'ROTPA     - sky_rotation')]:
    d = rot[col].dropna() if col in rot.columns else pd.Series(dtype=float)
    if len(d):
        print(f'\n{lbl}: n={len(d):4d}  mean={d.mean():+8.4f}  std={d.std():7.4f}  '
              f'max|d|={d.abs().max():7.4f} deg   '
              f'(|d|>{tol_deg}: {int((d.abs() > tol_deg).sum())})')
    else:
        print(f'\n{lbl}: no overlapping visits')

if output_file:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    out = Path(output_dir) / output_file
    rot.to_parquet(out, index=False)
    print(f'\nWrote {out}  ({len(rot)} rows)')
